Aqui resolveremos la siguiente pregunta:

P2: ¿Una estación estará en riesgo de quedarse sin bicicletas o sin anclajes disponibles en un periodo futuro?

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# Resolver la raíz del repositorio de forma agnóstica
ROOT_DIR = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Importar la ruta oficial de datos crudos centralizada
from proyecto_mibici.config import RAW_DATA_DIR

RAW_DIR = RAW_DATA_DIR

# Lista ordenada de los archivos mensuales crudos para la inspección
archivos = sorted(RAW_DIR.glob("datos_abiertos_*.csv"))

2026-10-18 06:21:47.670 | INFO     | proyecto_mibici.config:<module>:11 - PROJ_ROOT path is: /Users/antoniorios/Documents/maestria/trimestre 4/operaciones de aprendizaje/TC5061-Proyecto-Equipo20/TC5061-Proyecto-Equipo20-prev


In [3]:
# Lectura con la configuración por defecto de pandas (UTF-8)
for archivo in archivos:
    try:
        pd.read_csv(archivo, nrows=5)
        print(f"{archivo.name}: se abrió sin problema")
    except UnicodeDecodeError as error:
        print(f"{archivo.name}: NO se pudo abrir -> {type(error).__name__}: {error}")

datos_abiertos_2025_01.csv: NO se pudo abrir -> UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf1 in position 35: invalid continuation byte
datos_abiertos_2025_02.csv: NO se pudo abrir -> UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf1 in position 35: invalid continuation byte
datos_abiertos_2025_03.csv: se abrió sin problema
datos_abiertos_2025_04.csv: se abrió sin problema
datos_abiertos_2025_05.csv: se abrió sin problema
datos_abiertos_2025_06.csv: se abrió sin problema
datos_abiertos_2025_07.csv: NO se pudo abrir -> UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf1 in position 35: invalid continuation byte
datos_abiertos_2025_08.csv: se abrió sin problema
datos_abiertos_2025_09.csv: se abrió sin problema
datos_abiertos_2025_10.csv: se abrió sin problema
datos_abiertos_2025_11.csv: se abrió sin problema
datos_abiertos_2025_12.csv: se abrió sin problema
datos_abiertos_2026_01.csv: NO se pudo abrir -> UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf1 in p

De la celda anterior, podemos notar que no todos los archivos pueden abrirse simplemente con pd.read_csv. Tenemos el problema de:
UnicodeDecodeError. A continuación necesitamos saber donde está el problema.
Intuitivamente se piensa que el problema está en el encabezado debido a la posición del byte: 35. Corroboraremos lo anterior.

In [4]:
# Leer el encabezado de cada archivo con latin-1 (la codificación que sí abre los 18 archivos)
# Además, de momento no nos interesa leer los archivos completos, así que usamos nrows=5 y solo cargue las primeras 5 filas.
columnas_por_archivo = {}
for archivo in archivos:
    columnas = pd.read_csv(archivo, nrows=5, encoding="latin-1").columns.tolist()
    columnas_por_archivo[archivo.name] = columnas

# Todas las columnas que aparecen en algún archivo
todas = set()
for columnas in columnas_por_archivo.values():
    todas.update(columnas)  # Agrega elementos al conjunto


# Columnas que aparecen en TODOS los archivos
# Toma los valores del diccionario columnas_por_archivo y como cada valor es una lista entonces c es una lista
# pero cuando la colocamos en set(c), se olvida de que es una lista y mete a set solo los elementos de la lista.
# Después, el asterisco hace que en lugar de pasarle una lista de 18 conjuntos, le pasa 18 conjuntos
# separados.
# Luego, set.intersection nos devuelve la intsersección de los 18 conjuntos.
comunes = set.intersection(*[set(c) for c in columnas_por_archivo.values()])

print(f"Columnas distintas encontradas en total: {len(todas)}")
print(f"Columnas presentes en los {len(archivos)} archivos: {len(comunes)}")
print()
print("Columnas que NO están en todos los archivos:")
for columna in sorted(todas - comunes):                                                         # Sorted me devuelve una lista aunque sea una resta de conjuntos.
    archivos_con_columna = [n for n, c in columnas_por_archivo.items() if columna in c]         # items me devuelve tanto la palabra clave como su contenido en el diccionario
    print(f"  {columna!r}: aparece en {len(archivos_con_columna)} archivos")

# Nota:
#[  QUÉ GUARDAR   for  VARIABLES  in  DE DÓNDE   if  CONDICIÓN  ]
#[  n             for  n, c       in  columnas_por_archivo.items()   if  columna in c  ]           

Columnas distintas encontradas en total: 9
Columnas presentes en los 21 archivos: 7

Columnas que NO están en todos los archivos:
  'AÃ±o_de_nacimiento': aparece en 17 archivos
  'Año_de_nacimiento': aparece en 4 archivos


Con el resultado anterior, ahora sabemos que el problema está en el año de nacimiento. Además, no tenemos indicios de que exista el mismo problema en otra parte de los archivos.

In [5]:
inspeccion = []

for archivo in archivos:
    # ¿Se puede abrir en UTF-8?
    try:
        pd.read_csv(archivo, nrows=5, encoding="utf-8")
        abre_utf8 = True
    except UnicodeDecodeError:
        abre_utf8 = False

    # Columnas de este archivo (leídas con latin-1) que NO están en todos los archivos
    columnas = columnas_por_archivo[archivo.name]
    no_comunes = [col for col in columnas if col not in comunes]

    inspeccion.append({
        "archivo": archivo.name,
        "abre_utf8": abre_utf8,
        "n_columnas": len(columnas),
        "columna_distinta_latin1": ", ".join(no_comunes),
    })

inspeccion = pd.DataFrame(inspeccion)
print(inspeccion)
print()
print(pd.crosstab(inspeccion["abre_utf8"], inspeccion["columna_distinta_latin1"]))


                       archivo  abre_utf8  n_columnas columna_distinta_latin1
0   datos_abiertos_2025_01.csv      False           8       Año_de_nacimiento
1   datos_abiertos_2025_02.csv      False           8       Año_de_nacimiento
2   datos_abiertos_2025_03.csv       True           8      AÃ±o_de_nacimiento
3   datos_abiertos_2025_04.csv       True           8      AÃ±o_de_nacimiento
4   datos_abiertos_2025_05.csv       True           8      AÃ±o_de_nacimiento
5   datos_abiertos_2025_06.csv       True           8      AÃ±o_de_nacimiento
6   datos_abiertos_2025_07.csv      False           8       Año_de_nacimiento
7   datos_abiertos_2025_08.csv       True           8      AÃ±o_de_nacimiento
8   datos_abiertos_2025_09.csv       True           8      AÃ±o_de_nacimiento
9   datos_abiertos_2025_10.csv       True           8      AÃ±o_de_nacimiento
10  datos_abiertos_2025_11.csv       True           8      AÃ±o_de_nacimiento
11  datos_abiertos_2025_12.csv       True           8      AÃ±o_

La tabla cruzada muestra una correspondencia exacta: los 4 archivos que fallan en UTF-8 son los únicos en los que Latin-1 lee bien el nombre de la columna, y los 14 que abren en UTF-8 son los que producen mojibake con Latin-1. Cada uno se lee bien con su propia codificación. El problema es que los archivos no comparten una misma codificación.


In [6]:
dataframes = []
resumen_lectura = []

for archivo in archivos:
    # Leer el archivo completo: primero UTF-8 y, si falla, Latin-1
    try:
        df = pd.read_csv(archivo, encoding="utf-8")
        codificacion = "utf-8"
    except UnicodeDecodeError:
        df = pd.read_csv(archivo, encoding="latin-1")
        codificacion = "latin-1"

    # Etiqueta de origen: de qué archivo viene cada fila (ej. "2025_01")
    df["periodo"] = archivo.stem.replace("datos_abiertos_", "")

    dataframes.append(df)
    resumen_lectura.append({
        "archivo": archivo.name,
        "codificacion": codificacion,
        "filas": len(df),
        "columnas": df.shape[1],
    })

# Unir los 18 archivos en un solo DataFrame
viajes = pd.concat(dataframes, ignore_index=True)

# Resumen de la lectura
resumen_lectura = pd.DataFrame(resumen_lectura)
print(resumen_lectura)
print()
print(resumen_lectura["codificacion"].value_counts())
print()

# Verificaciones de la unión
print(f"Filas en el DataFrame unido:      {len(viajes):,}")
print(f"Suma de filas de los 18 archivos: {resumen_lectura['filas'].sum():,}")
print(f"Columnas ({viajes.shape[1]}): {viajes.columns.tolist()}")

del dataframes  # ya no se necesita; libera memoria

                       archivo codificacion   filas  columnas
0   datos_abiertos_2025_01.csv      latin-1  380184         9
1   datos_abiertos_2025_02.csv      latin-1  372529         9
2   datos_abiertos_2025_03.csv        utf-8  405110         9
3   datos_abiertos_2025_04.csv        utf-8  366193         9
4   datos_abiertos_2025_05.csv        utf-8  395891         9
5   datos_abiertos_2025_06.csv        utf-8  356611         9
6   datos_abiertos_2025_07.csv      latin-1  368186         9
7   datos_abiertos_2025_08.csv        utf-8  374727         9
8   datos_abiertos_2025_09.csv        utf-8  376789         9
9   datos_abiertos_2025_10.csv        utf-8  420064         9
10  datos_abiertos_2025_11.csv        utf-8  380328         9
11  datos_abiertos_2025_12.csv        utf-8  335420         9
12  datos_abiertos_2026_01.csv      latin-1  357180         9
13  datos_abiertos_2026_02.csv        utf-8  333756         9
14  datos_abiertos_2026_03.csv        utf-8  376243         9
15  dato

Cada archivo se leyó primero con UTF-8, que falla de forma explícita si la codificación no corresponde, y, si falla, con Latin-1. 
De esta forma 14 archivos se leyeron en UTF-8 y 4 en Latin-1. Al unirlos se obtienen 6,622,260 viajes, que coinciden con la suma de los 18 archivos, y 9 columnas (las 8 originales más periodo), con un solo nombre para la columna del año de nacimiento.

In [7]:
viajes.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7894097 entries, 0 to 7894096
Data columns (total 9 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   Viaje_Id           7894097 non-null  int64  
 1   Usuario_Id         7894097 non-null  int64  
 2   Genero             7831354 non-null  object 
 3   Año_de_nacimiento  7892661 non-null  float64
 4   Inicio_del_viaje   7894097 non-null  object 
 5   Fin_del_viaje      7894097 non-null  object 
 6   Origen_Id          7894097 non-null  int64  
 7   Destino_Id         7894097 non-null  int64  
 8   periodo            7894097 non-null  object 
dtypes: float64(1), int64(4), object(4)
memory usage: 542.0+ MB


Tipos de datos. Los identificadores (Viaje_Id, Usuario_Id, Origen_Id, Destino_Id) son enteros, lo cual es correcto. Hay dos tipos que no corresponden a lo que representa la columna:
Inicio_del_viaje y Fin_del_viaje son fechas, pero pandas las leyó como object. Así no se pueden calcular duraciones ni extraer la hora o el día.
Año_de_nacimiento es un año, pero pandas lo leyó como decimal (float64).
Así sabemos que hay faltantes en Año de nacimiento y Genero

In [8]:
print(viajes[["Inicio_del_viaje", "Fin_del_viaje", "Año_de_nacimiento"]].sample(5, random_state=42))
print()
print(f"Faltantes en Año_de_nacimiento: {viajes['Año_de_nacimiento'].isna().sum():,}")

            Inicio_del_viaje        Fin_del_viaje  Año_de_nacimiento
2968874  2025-08-27 17:29:42  2025-08-27 17:36:52             1995.0
123767   2025-01-13 13:54:12  2025-01-13 14:01:23             1999.0
5357766  2026-03-11 16:12:30  2026-03-11 16:41:52             1990.0
7200202  2026-08-16 15:45:08  2026-08-16 15:57:10             1994.0
1391027  2025-04-19 22:14:31  2025-04-19 22:25:44             1995.0

Faltantes en Año_de_nacimiento: 1,436


Lo importante es que hemos visto de manera aleatoria. Como Año de nacimiento es un entero, no debe haber decimales. Además, vemos que hay 1315 faltantes. Mientras que Inicio del Viaje y Fin del viaje tienen el formato: AAAA-MM-DD HH:MM:SS. Veamos si todas las fechas tienen ese formato y ver que de verdad no haya decimales en Año de nacimiento.

In [9]:
# Hipótesis 1: todas las fechas tienen el formato AAAA-MM-DD HH:MM:SS
for col in ["Inicio_del_viaje", "Fin_del_viaje"]:
    convertidas = pd.to_datetime(viajes[col], format="%Y-%m-%d %H:%M:%S", errors="coerce")  
    print(f"{col}")
    print(f"  Largo del texto (caracteres: filas): {viajes[col].str.len().value_counts().to_dict()}")
    print(f"  No convertibles con ese formato: {convertidas.isna().sum()}")
    print(f"  Rango: {convertidas.min()}  a  {convertidas.max()}")

# Hipótesis 2: ningún año tiene parte decimal
anio = viajes["Año_de_nacimiento"].dropna()
print(f"\nAños con parte decimal distinta de cero: {(anio % 1 != 0).sum()}")

# coerce: si un valor no encaja en el formato, en lugar de detener la celda lo convierte en NaT (faltante de fecha). Así podemos contar cuántos fallan.
# .str.len() mide cuántos caracteres tiene cada texto. Si el formato es uniforme, todas deben medir lo mismo. Es una segunda comprobación, independiente de la primera.

Inicio_del_viaje
  Largo del texto (caracteres: filas): {19: 7894097}
  No convertibles con ese formato: 0
  Rango: 2025-01-01 00:00:44  a  2026-09-30 23:59:48
Fin_del_viaje
  Largo del texto (caracteres: filas): {19: 7894097}
  No convertibles con ese formato: 0
  Rango: 2025-01-01 00:11:29  a  2026-10-01 00:14:52

Años con parte decimal distinta de cero: 0


Se confirman las dos hipótesis en las 6,622,260 filas:

Las fechas de inicio y fin miden 19 caracteres y todas se convierten con el formato AAAA-MM-DD HH:MM:SS, sin ningún fallo.
Ningún año de nacimiento tiene parte decimal: el float64 se debe solo a los faltantes.

Los rangos coinciden con el periodo de los archivos (enero de 2025 a junio de 2026). Hay una observación pendiente: Fin_del_viaje llega hasta el 1 de julio de 2026.
A partir de la siguiente celda, crearemos una copia de viajes.

In [10]:
viajes_limpio = viajes.copy()

viajes_limpio["Inicio_del_viaje"] = pd.to_datetime(viajes_limpio["Inicio_del_viaje"], format="%Y-%m-%d %H:%M:%S")
viajes_limpio["Fin_del_viaje"] = pd.to_datetime(viajes_limpio["Fin_del_viaje"], format="%Y-%m-%d %H:%M:%S")
viajes_limpio["Año_de_nacimiento"] = viajes_limpio["Año_de_nacimiento"].astype("Int64")

# Verificación: los no nulos deben ser los mismos antes y después
comparacion = pd.DataFrame({
    "tipo_antes": viajes.dtypes,
    "tipo_despues": viajes_limpio.dtypes,
    "no_nulos_antes": viajes.notna().sum(),
    "no_nulos_despues": viajes_limpio.notna().sum(),
})
print(comparacion)

                  tipo_antes    tipo_despues  no_nulos_antes  no_nulos_despues
Viaje_Id               int64           int64         7894097           7894097
Usuario_Id             int64           int64         7894097           7894097
Genero                object          object         7831354           7831354
Año_de_nacimiento    float64           Int64         7892661           7892661
Inicio_del_viaje      object  datetime64[ns]         7894097           7894097
Fin_del_viaje         object  datetime64[ns]         7894097           7894097
Origen_Id              int64           int64         7894097           7894097
Destino_Id             int64           int64         7894097           7894097
periodo               object          object         7894097           7894097


Se convierten las fechas a datetime y el año de nacimiento a Int64 (entero que admite faltantes), en una copia (viajes_limpio) para conservar los datos originales. La tabla comparativa muestra que el número de valores no nulos es el mismo antes y después en todas las columnas, así que la conversión no perdió ni alteró ningún dato.

In [11]:
nombres_nuevos = {
    "Viaje_Id": "viaje_id",
    "Usuario_Id": "usuario_id",
    "Genero": "genero",
    "Año_de_nacimiento": "anio_nacimiento",
    "Inicio_del_viaje": "inicio",
    "Fin_del_viaje": "fin",
    "Origen_Id": "origen_id",
    "Destino_Id": "destino_id",
}

viajes_limpio = viajes_limpio.rename(columns=nombres_nuevos)

print(viajes_limpio.columns.tolist())
print(f"Columnas con caracteres no ASCII: {[c for c in viajes_limpio.columns if not c.isascii()]}")

['viaje_id', 'usuario_id', 'genero', 'anio_nacimiento', 'inicio', 'fin', 'origen_id', 'destino_id', 'periodo']
Columnas con caracteres no ASCII: []


En la celda anterior simplemente homologamos los nombres quitando la ñ para evitar posibles errores y renombramos otras variables para facilitar todo de aqui en adelante.

A continuación trataremos el tema de los faltantes.

In [12]:
faltantes = pd.DataFrame({
    "faltantes": viajes_limpio.isna().sum(),
    "porcentaje": (viajes_limpio.isna().mean() * 100).round(3),
})
print(faltantes)

                 faltantes  porcentaje
viaje_id                 0       0.000
usuario_id               0       0.000
genero               62743       0.795
anio_nacimiento       1436       0.018
inicio                   0       0.000
fin                      0       0.000
origen_id                0       0.000
destino_id               0       0.000
periodo                  0       0.000


Faltantes. Solo dos columnas tienen faltantes: genero (54,465; 0.82 %) y anio_nacimiento (1,315; 0.02 %). Las columnas que definen el viaje (identificadores, fechas y estaciones de origen y destino) están completas.

In [13]:
print(viajes_limpio["genero"].value_counts(dropna=False))

genero
M      5674158
F      2157196
NaN      62743
Name: count, dtype: int64


genero solo toma dos valores válidos, M y F, además de los faltantes. No hay variantes mal escritas (como m, Masculino o espacios).

Hasta ahora, los faltantes los detectó pandas al leer, pero no sabemos cómo venían escritos en los archivos. Conviene saberlo: si dataset.py depende de que pandas los reconozca solo, podría dejar de hacerlo con otra versión o con otra forma de escribirlos. Se revisa el texto original.

In [14]:
# Codificación de cada archivo, tomada de la lectura anterior
codificacion_por_archivo = dict(zip(resumen_lectura["archivo"], resumen_lectura["codificacion"]))

columnas = ["Genero", "Año_de_nacimiento"]   # las dos columnas con faltantes (paso 1)
marcadores = []

for archivo in archivos:
    cod = codificacion_por_archivo[archivo.name]

    # Lectura normal: pandas convierte los faltantes en NaN
    interpretado = pd.read_csv(archivo, encoding=cod, usecols=columnas)
    # Lectura como texto puro: pandas no convierte nada
    texto = pd.read_csv(archivo, encoding=cod, usecols=columnas, dtype=str, keep_default_na=False)

    fila = {"archivo": archivo.name}
    for col in columnas:
        # Qué texto había en el archivo en las filas que pandas marcó como faltantes
        fila[col] = texto.loc[interpretado[col].isna(), col].value_counts().to_dict()
    marcadores.append(fila)

marcadores = pd.DataFrame(marcadores)
print(marcadores.to_string())

                       archivo          Genero Año_de_nacimiento
0   datos_abiertos_2025_01.csv  {'NULL': 2702}       {'NA': 147}
1   datos_abiertos_2025_02.csv  {'NULL': 2982}       {'NA': 133}
2   datos_abiertos_2025_03.csv  {'NULL': 3213}       {'NA': 176}
3   datos_abiertos_2025_04.csv  {'NULL': 3483}       {'NA': 160}
4   datos_abiertos_2025_05.csv  {'NULL': 3706}        {'NA': 96}
5   datos_abiertos_2025_06.csv  {'NULL': 3333}        {'NA': 98}
6   datos_abiertos_2025_07.csv  {'NULL': 3464}        {'NA': 38}
7   datos_abiertos_2025_08.csv  {'NULL': 3452}        {'NA': 25}
8   datos_abiertos_2025_09.csv  {'NULL': 3426}        {'NA': 29}
9   datos_abiertos_2025_10.csv  {'NULL': 4343}        {'NA': 33}
10  datos_abiertos_2025_11.csv  {'NULL': 3894}        {'NA': 65}
11  datos_abiertos_2025_12.csv  {'NULL': 2791}       {'NA': 105}
12  datos_abiertos_2026_01.csv  {'NULL': 2616}        {'NA': 90}
13  datos_abiertos_2026_02.csv  {'NULL': 2400}        {'NA': 41}
14  datos_abiertos_2026_0

Comparando el texto original con lo que pandas interpretó como faltante: en Genero, todos los faltantes vienen como el texto NULL, y en Año_de_nacimiento, como NA. Cada columna usa un marcador distinto, y pandas los reconoce porque ambos están en su lista de valores faltantes por defecto.

In [15]:
columnas_numericas = ["viaje_id", "usuario_id", "anio_nacimiento", "origen_id", "destino_id"]
print(viajes_limpio[columnas_numericas].describe().round(1).to_string())

         viaje_id  usuario_id  anio_nacimiento  origen_id  destino_id
count   7894097.0   7894097.0        7892661.0  7894097.0   7894097.0
mean   41698615.2   2582881.8           1990.5      160.0       161.8
std     2627989.7   1766340.9             15.9      117.9       120.6
min    37162342.0       102.0            199.0        2.0         2.0
25%    39420223.0    652600.0           1985.0       51.0        51.0
50%    41686211.0   2554316.0           1993.0      157.0       157.0
75%    43948440.0   4516124.0           1999.0      260.0       266.0
max    46255558.0   5006315.0           2023.0      519.0       519.0


Rangos de las columnas numéricas. Los identificadores de viaje, usuario y estación son enteros positivos, sin ceros ni negativos. Las estaciones de origen y destino van del 2 al 503, rango que se comparará después con el catálogo de estaciones.

En anio_nacimiento hay dos valores sospechosos: el mínimo es 199, que no es un año posible, y el máximo es 2023, que implicaría que un niño de 2 o 3 años usó el servicio en 2025 o 2026. Los cuartiles (1985, 1993 y 1999) indican que la gran mayoría de los valores son razonables, así que se revisan los extremos para saber cuántos viajes están afectados y dónde termina el rango plausible.

In [16]:
conteo_anios = viajes_limpio["anio_nacimiento"].value_counts().sort_index()

print("Años más bajos:")
print(conteo_anios.head(15).to_string())
print("\nAños más altos:")
print(conteo_anios.tail(15).to_string())

Años más bajos:
anio_nacimiento
199      316
1420       3
1920      30
1939     377
1941     518
1942    1809
1943       1
1944      50
1945    1133
1946     185
1947      61
1948    2152
1949    6759
1950    1310
1951     844

Años más altos:
anio_nacimiento
1997    399728
1998    354691
1999    386907
2000    357011
2001    348960
2002    288100
2003    214218
2004    175758
2005    120292
2006    100570
2007     58374
2008     23799
2009      4984
2010       918
2023       398


El 199 y el 2023 no son errores aislados de un solo viaje: aparecen en 278 y 398 viajes, respectivamente. Además, no son los únicos años sospechosos: 1920 (30 viajes, implicaría una edad de 105 años) queda separado del resto por un hueco de casi 20 años, ya que el siguiente año registrado es 1939. En el otro extremo, 2023 queda separado por 13 años del anterior (2010). Entre 1939 y 2010 los años forman una secuencia continua, lo que sugiere que ese es el rango plausible (a excepción de 1940).

Decisión sobre genero y anio_nacimiento. Estas dos columnas describen al usuario y concentran los problemas de calidad encontrados: faltantes (0.82 % y 0.02 %) y años de nacimiento inválidos (199 y 2023, este último aislado del resto del rango, que termina en 2010). Para P2 no se utilizan, porque el riesgo de vaciado o llenado de una estación depende de cuántas bicicletas salen y llegan, no de quién las usa. Esto además aplica el principio de minimización de datos personales. No se eliminan filas por estos problemas, porque son viajes válidos que sí cuentan para el flujo de las estaciones.

In [17]:
# 1. Filas idénticas en todas las columnas
print(f"Filas completamente duplicadas: {viajes_limpio.duplicated().sum():,}")

# 2. Identificadores de viaje repetidos
print(f"viaje_id repetidos: {viajes_limpio['viaje_id'].duplicated().sum():,}")

# 3. El mismo viaje con distinto identificador: todo igual excepto viaje_id
columnas_sin_id = [c for c in viajes_limpio.columns if c != "viaje_id"]
print(f"Filas duplicadas ignorando viaje_id: {viajes_limpio.duplicated(subset=columnas_sin_id).sum():,}")

Filas completamente duplicadas: 758
viaje_id repetidos: 758
Filas duplicadas ignorando viaje_id: 815


De la búsqueda de duplicados observamos lo siguiente:
No existen filas duplicadas
Tampoco existen ID de viaje duplicados
Buscamos que toda la fila fuera exactamente igual, pero con un ID distinto. Lo anterior es casi imposible porque sería el mismo usuario sacando dos bicis en el mismo segundo, de la misma estación y hacia el mismo destino, y devolviéndolas en el mismo segundo.

In [18]:
mascara = viajes_limpio.duplicated(subset=columnas_sin_id, keep=False)
print(viajes_limpio[mascara].sort_values(["usuario_id", "inicio"]).head(16).to_string())

         viaje_id  usuario_id genero  anio_nacimiento              inicio                 fin  origen_id  destino_id  periodo
7426268  45731330       21150      M             1982 2026-09-02 15:32:53 2026-09-02 15:38:36         75          68  2026_09
7426269  45731330       21150      M             1982 2026-09-02 15:32:53 2026-09-02 15:38:36         75          68  2026_09
7428886  45734267       21150      M             1982 2026-09-02 17:50:26 2026-09-02 17:56:50         68          75  2026_09
7428887  45734267       21150      M             1982 2026-09-02 17:50:26 2026-09-02 17:56:50         68          75  2026_09
7553202  45877315       44000      M             1985 2026-09-11 17:27:42 2026-09-11 17:34:36         26          81  2026_09
7553203  45877315       44000      M             1985 2026-09-11 17:27:42 2026-09-11 17:34:36         26          81  2026_09
7532720  45854187       45350      M             1981 2026-09-10 11:42:59 2026-09-10 11:49:17         25          20  

In [19]:
columnas_sin_id = [c for c in viajes_limpio.columns if c != "viaje_id"]
mascara = viajes_limpio.duplicated(subset=columnas_sin_id, keep=False)

grupos = (
    viajes_limpio[mascara]
    .groupby(columnas_sin_id, dropna=False)
    .agg(copias=("viaje_id", "size"), id_min=("viaje_id", "min"), id_max=("viaje_id", "max"))
    .reset_index()
)
grupos["duracion_seg"] = (grupos["fin"] - grupos["inicio"]).dt.total_seconds()

print(f"Filas involucradas: {mascara.sum()}  |  Viajes distintos: {len(grupos)}  |  Copias de más: {(grupos['copias'] - 1).sum()}")
print()
print(grupos[["usuario_id", "inicio", "origen_id", "destino_id", "copias", "duracion_seg", "id_min", "id_max", "periodo"]]
      .sort_values("inicio").to_string(index=False))

Filas involucradas: 1621  |  Viajes distintos: 806  |  Copias de más: 815

 usuario_id              inicio  origen_id  destino_id  copias  duracion_seg   id_min   id_max periodo
    4545076 2026-05-21 15:01:59        102         102       2           2.0 44267943 44267970 2026_05
    4037245 2026-05-24 15:37:31        169         169       2           2.0 44308494 44308500 2026_05
    3352022 2026-05-27 07:03:24        289         289       2           2.0 44344472 44344475 2026_05
    4036704 2026-05-29 08:36:49        102         103       2           2.0 44378243 44378244 2026_05
    2512064 2026-05-29 15:35:18         42          68       2           2.0 44383385 44383386 2026_05
    4704227 2026-05-31 22:04:52        416          84       2           2.0 44407716 44407717 2026_05
    4650176 2026-06-03 10:21:50        239         239       2           2.0 44445466 44445852 2026_06
     705721 2026-06-05 06:50:54        161          45       2           2.0 44466163 44466166 2026_0

No hay filas idénticas ni identificadores de viaje repetidos. Sin embargo, al ignorar viaje_id aparecen 21 viajes registrados más de una vez (48 filas, 27 copias de más), con identificadores casi consecutivos. Los 21 tienen una característica común: duran exactamente 2 segundos, incluso entre estaciones distintas, lo cual es físicamente imposible. Además, todos ocurren en mayo y junio de 2026, ninguno antes. Esto sugiere que no son viajes reales, sino registros generados por el sistema, posiblemente por un cambio a partir de mayo de 2026.

Las 27 copias son pocas (0.0004 % de los viajes), pero el hallazgo abre una pregunta más amplia: ¿cuántos viajes, duplicados o no, tienen duraciones imposibles? La decisión sobre estos registros se toma después de revisar la duración de todos los viajes.

Lo anterior nos lleva a duda: ¿cuánto duran en general los viajes?

In [20]:
viajes_limpio["duracion_min"] = (viajes_limpio["fin"] - viajes_limpio["inicio"]).dt.total_seconds() / 60

print(viajes_limpio["duracion_min"]
      .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
      .round(2)
      .to_string())

count    7894097.00
mean          11.04
std           10.44
min            0.00
1%             0.20
5%             2.20
25%            5.73
50%            9.45
75%           14.80
95%           24.38
99%           33.18
max        11500.77


Duración de los viajes. Se calcula la duración de cada viaje como fin − inicio, en minutos.

Viaje típico: la mediana es de 9.58 minutos, y la mitad central de los viajes (del percentil 25 al 75) dura entre 5.8 y 15 minutos. El 99 % de los viajes dura menos de 34 minutos. Es un patrón coherente con un sistema de bicicletas para trayectos cortos dentro de la ciudad.
Asimetría: la media (11.16 min) es mayor que la mediana (9.58 min), lo que indica una cola de viajes largos que jala el promedio hacia arriba.
No hay duraciones negativas: el mínimo es 0, así que ningún viaje termina antes de empezar.

Sin embargo, las dos puntas son sospechosas:

Viajes demasiado cortos: el 1 % de los viajes dura menos de 0.23 minutos (unos 14 segundos), y el 5 % menos de 2.3 minutos. En 14 segundos difícilmente se puede llegar a otra estación, y los viajes duplicados encontrados antes duraban justo 2 segundos.
Viajes demasiado largos: el máximo es de 11,501 minutos, casi 8 días. Un trayecto urbano no dura eso; puede ser una bicicleta que no se ancló bien o un error de registro.

Se revisan las dos puntas por separado: los viajes cortos en segundos, para ver si hay un umbral natural, y los largos, para ver cuántos superan duraciones razonables.

In [21]:
viajes_limpio["duracion_seg"] = (viajes_limpio["fin"] - viajes_limpio["inicio"]).dt.total_seconds()
viajes_limpio["misma_estacion"] = viajes_limpio["origen_id"] == viajes_limpio["destino_id"]

# Viajes de menos de 2 minutos, agrupados en tramos de 10 segundos
cortos = viajes_limpio[viajes_limpio["duracion_seg"] < 120]
tramos = pd.cut(cortos["duracion_seg"], bins=range(0, 121, 10), right=False)

tabla_cortos = pd.crosstab(tramos, cortos["misma_estacion"], margins=True, margins_name="total")
tabla_cortos.columns = ["distinta_estacion", "misma_estacion", "total"]
print(tabla_cortos.to_string())

              distinta_estacion  misma_estacion   total
duracion_seg                                           
[0, 10)                   20362           43986   64348
[10, 20)                      1           45899   45900
[20, 30)                      0           76918   76918
[30, 40)                     71           24760   24831
[40, 50)                    289           18705   18994
[50, 60)                    841           13485   14326
[60, 70)                   2274           10200   12474
[70, 80)                   4884            7290   12174
[80, 90)                   8981            5576   14557
[90, 100)                 14092            4401   18493
[100, 110)                19755            3634   23389
[110, 120)                24968            2929   27897
total                     96518          257783  354301


Se escogieron intervalos de 10 segundos solo por exploración y no viene de una decisión tomada con base en algún hallazgo. Sin embargo, el output resultó ser bastante interesante:

Lo más importante a destacar de la celda anterior es el hecho de que existen 15,954 viajes cuyo destino es distinto a la estación de origen; pero el viaje dura menos de 10 segundos, lo cual suena ilógico. De hecho, parece que las cifras de viajes a distintas estaciones toman sentido a partir del intervalo [30 a 40) segundos.

Por otro lado, cuando los viajes son en la misma estación los viajes deben durar poco porque tal vez solo den una vuelta rápida o incluso se arrepienta el usuario del viaje. Por lo tanto, tiene sentido que su duración sea relativamente corta.

Una pregunta adicional a hacerse es la siguiente: Los duplicados de 2 segundos estaban todos en mayo y junio de 2026. ¿Pasa lo mismo con estos 15,954 viajes imposibles, o están repartidos en todos los meses?

In [22]:
imposibles = viajes_limpio[(viajes_limpio["duracion_seg"] < 10) & (~viajes_limpio["misma_estacion"])]
print(imposibles["periodo"].value_counts().sort_index().to_string())

periodo
2025_01     432
2025_02     421
2025_03     534
2025_04     365
2025_05     452
2025_06     711
2025_07     488
2025_08     528
2025_09     536
2025_10     489
2025_11    1213
2025_12    1217
2026_01    1605
2026_02    1330
2026_03    1144
2026_04     596
2026_05     536
2026_06    3357
2026_07    1581
2026_08    1692
2026_09    1135


¿Cuándo ocurren los viajes imposibles? Los 15,954 viajes de menos de 10 segundos entre estaciones distintas aparecen en los 18 meses, así que no son un problema puntual. Sin embargo, su frecuencia no es constante:

De enero a octubre de 2025 se mantienen entre 365 y 711 por mes.
A partir de noviembre de 2025 aumentan a más de 1,100 por mes y llegan a 1,605 en enero de 2026.
En abril y mayo de 2026 bajan a unos 550.
En junio de 2026 alcanzan el máximo, 3,357, unas cinco veces más que un mes típico de 2025.

Como el número total de viajes por mes es parecido (entre 315 y 420 mil), estas variaciones no se explican por el volumen: reflejan cambios en la frecuencia del error. Dicha información viene de la salida de resumen_lectura.

Relación con los duplicados: los duplicados de 2 segundos estaban solo en mayo y junio de 2026, pero los viajes imposibles existen desde enero de 2025. Los duplicados son, por lo tanto, una parte de un problema más amplio y persistente: el sistema registra movimientos entre estaciones que no pudieron ocurrir.

Implicación para P2: como el problema es más frecuente en 2026, si no se filtra, los conjuntos de validación (enero a marzo de 2026) y prueba (abril a junio de 2026) tendrían más salidas y llegadas falsas que el de entrenamiento (2025). En particular, junio de 2026, que forma parte de la prueba, es el mes más afectado. Para que los tres conjuntos sean comparables, la misma regla de limpieza debe aplicarse a todos los meses.

A continuación, ¿cuántos viajes duran más de lo razonable? Sabemos que al menos hay un viaje que duro 11,500 minutos; lo cual es imposible.

In [23]:
percentiles_extremos = [0.99, 0.995, 0.999, 0.9995, 0.9999, 0.99999, 1.0]
print(viajes_limpio["duracion_min"].quantile(percentiles_extremos).round(1).to_string())

0.99000       33.2
0.99500       38.6
0.99900       58.2
0.99950       77.8
0.99990      190.6
0.99999      874.1
1.00000    11500.8


Viajes largos. Los percentiles extremos muestran que la cola de viajes largos es muy larga, pero muy delgada:

Hasta el percentil 99.9, la duración crece despacio: de 33 minutos (p99) a 58 minutos (p99.9). Es decir, el 99.9 % de los viajes dura menos de una hora.
A partir de ahí crece muy rápido: el percentil 99.99 es de 188 minutos (más de 3 horas), el 99.999 de 856 minutos (más de 14 horas) y el máximo de 11,501 minutos (casi 8 días).

Con 6.6 millones de viajes, el 0.1 % equivale a unos 6,600 viajes de más de una hora, y el 0.01 % a unos 660 de más de 3 horas. Una duración de varias horas o días no corresponde a un trayecto urbano: probablemente son bicicletas que no se anclaron correctamente o registros que se cerraron tarde.

In [24]:
umbrales_min = [30, 60, 120, 180, 360, 720, 1440]

largos = pd.DataFrame({
    "mas_de_min": umbrales_min,
    "viajes": [(viajes_limpio["duracion_min"] > u).sum() for u in umbrales_min],
})
largos["porcentaje"] = (largos["viajes"] / len(viajes_limpio) * 100).round(3)
print(largos.to_string(index=False))

print("\nLos 5 viajes más largos:")
print(viajes_limpio.nlargest(5, "duracion_min")[["inicio", "fin", "origen_id", "destino_id", "duracion_min"]].to_string())

 mas_de_min  viajes  porcentaje
         30  126481       1.602
         60    7255       0.092
        120    1698       0.022
        180     854       0.011
        360     329       0.004
        720     127       0.002
       1440      17       0.000

Los 5 viajes más largos:
                     inicio                 fin  origen_id  destino_id  duracion_min
2504446 2025-07-20 08:35:05 2025-07-28 08:15:51         48          72  11500.766667
5194638 2026-02-26 15:33:25 2026-03-02 18:13:28        263         258   5920.050000
6898007 2026-07-22 10:08:48 2026-07-25 13:51:49        420         420   4543.016667
5530593 2026-03-25 20:12:10 2026-03-27 12:25:01        230         256   2412.850000
6320343 2026-06-05 07:23:51 2026-06-06 20:24:38        371         321   2220.783333


Como complemento, se cuentan los viajes que superan duraciones de referencia fáciles de interpretar (media hora, 1, 2, 3, 6 y 12 horas, y 1 día). Estos cortes se eligieron por legibilidad, no a partir de un hallazgo. Confirman lo que mostraron los percentiles: 5,961 viajes (0.09 %) duran más de una hora, 703 (0.011 %) más de 3 horas y 15 más de un día.

Los 5 viajes más largos van de casi 37 horas a 8 días, todos entre estaciones distintas. Que una bicicleta tarde días en llegar a otra estación no corresponde a un trayecto real: lo más probable es que la bicicleta no se haya anclado bien o que el registro se haya cerrado tarde.

Decisión sobre las duraciones y los duplicados. Para P2 no importa si un viaje es razonable como trayecto, sino si refleja un movimiento real de bicicletas entre estaciones, porque eso es lo que cambia su inventario. Con ese criterio:

Se eliminan los viajes de menos de 10 segundos entre estaciones distintas: ninguna bicicleta pudo moverse de una estación a otra en ese tiempo, así que registrarían salidas y llegadas que no ocurrieron.
Se eliminan las copias de los viajes duplicados, conservando un registro de cada uno.
Se conservan los viajes largos: aunque no tengan sentido como trayecto, la bicicleta sí estuvo fuera de la estación de origen todo ese tiempo y sí llegó a la de destino. Además, son pocos (0.09 % duran más de una hora).
Se conservan los viajes cortos en la misma estación: la bicicleta salió y volvió a la misma estación, así que su efecto en el balance es nulo.

La misma regla se aplica a los 18 meses, para que los conjuntos de entrenamiento, validación y prueba sean comparables.

In [25]:
columnas_sin_id = ["usuario_id", "genero", "anio_nacimiento", "inicio", "fin", "origen_id", "destino_id", "periodo"]

es_copia = viajes_limpio.duplicated(subset=columnas_sin_id)
es_imposible = (viajes_limpio["duracion_seg"] < 10) & (~viajes_limpio["misma_estacion"])
a_eliminar = es_copia | es_imposible

print(f"Copias de viajes duplicados:            {es_copia.sum():,}")
print(f"Viajes imposibles (< 10 s, distinta est.): {es_imposible.sum():,}")
print(f"Cumplen ambas condiciones:               {(es_copia & es_imposible).sum():,}")
print(f"Total a eliminar:                        {a_eliminar.sum():,}")

filas_antes = len(viajes_limpio)
viajes_limpio = viajes_limpio[~a_eliminar].reset_index(drop=True)
print(f"\nViajes antes: {filas_antes:,}  |  después: {len(viajes_limpio):,}  |  eliminados: {filas_antes - len(viajes_limpio):,} ({(filas_antes - len(viajes_limpio)) / filas_antes * 100:.2f} %)")

Copias de viajes duplicados:            815
Viajes imposibles (< 10 s, distinta est.): 20,362
Cumplen ambas condiciones:               44
Total a eliminar:                        21,133

Viajes antes: 7,894,097  |  después: 7,872,964  |  eliminados: 21,133 (0.27 %)


Se eliminaron 15,964 viajes (0.24 %): 15,954 viajes imposibles y 27 copias de duplicados, de las cuales 17 eran también viajes imposibles, por lo que no se cuentan dos veces. Quedan 6,606,296 viajes para el análisis de P2.

Ahora, recordemos que existen datos para el 1 de julio de 2026. Lo cual excede el periodo del test que es de abril a junio de 2026.
Arreglemos eso.

In [26]:
mes_inicio = viajes_limpio["inicio"].dt.strftime("%Y_%m")
mes_fin = viajes_limpio["fin"].dt.strftime("%Y_%m")

print(f"Viajes que empiezan fuera del mes de su archivo: {(mes_inicio != viajes_limpio['periodo']).sum():,}")
print(f"Viajes que terminan en otro mes:                 {(mes_fin != viajes_limpio['periodo']).sum():,}")

cruzan_mes = viajes_limpio[mes_fin != viajes_limpio["periodo"]]
print("\nDuración (min) de los viajes que terminan en otro mes:")
print(cruzan_mes["duracion_min"].describe().round(1).to_string())

print("\nViajes que terminan en otro mes, por archivo:")
print(cruzan_mes["periodo"].value_counts().sort_index().to_string())

Viajes que empiezan fuera del mes de su archivo: 0
Viajes que terminan en otro mes:                 549

Duración (min) de los viajes que terminan en otro mes:
count     549.0
mean       40.7
std       270.1
min         2.4
25%        10.4
50%        16.4
75%        21.9
max      5920.0

Viajes que terminan en otro mes, por archivo:
periodo
2025_01    24
2025_02    38
2025_03    20
2025_04    26
2025_05    36
2025_06    18
2025_07    37
2025_08     7
2025_09    33
2025_10    24
2025_11    16
2025_12     7
2026_01    21
2026_02    21
2026_03    21
2026_04    42
2026_05    15
2026_06    75
2026_07    34
2026_08    18
2026_09    16


Consistencia de las fechas. Ningún viaje empieza fuera del mes de su archivo. En cambio, 481 viajes terminan en el mes siguiente, y esto ocurre en los 18 meses, no solo en junio de 2026. En su mayoría tienen duración normal (mediana de 16.4 minutos), lo que es coherente con viajes que cruzan la medianoche del cambio de mes, así que no son errores: son viajes que empiezan al final de un mes y terminan después de la medianoche. Por lo tanto, los viajes de junio de 2026 que terminan el 1 de julio siguen el mismo patrón que el resto de los meses y se conservan. Esto explica la observación pendiente sobre el rango de fin.

Al agregar por estación y hora, las horas del 1 de julio de 2026 quedarán fuera del periodo de análisis, porque solo contendrían llegadas. Por lo tanto, no hay nada que arreglar al respecto.

A continuación analizaremos la nomenclatura de la base con el archivo nomenclatura_2026_09.

In [27]:
archivo_estaciones = RAW_DIR / "nomenclatura_2026_09.csv"

try:
    estaciones = pd.read_csv(archivo_estaciones, encoding="utf-8")
    codificacion_estaciones = "utf-8"
except UnicodeDecodeError:
    estaciones = pd.read_csv(archivo_estaciones, encoding="latin-1")
    codificacion_estaciones = "latin-1"

print(f"Codificación: {codificacion_estaciones}")
print(f"Filas: {len(estaciones)}  |  Columnas: {estaciones.shape[1]}")
print()
estaciones.info(show_counts=True)
print()
print(estaciones.head().to_string())

Codificación: utf-8
Filas: 484  |  Columnas: 5

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 484 entries, 0 to 483
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          484 non-null    int64  
 1   name        484 non-null    object 
 2   latitude    484 non-null    float64
 3   longitude   484 non-null    float64
 4   dpcapacity  484 non-null    int64  
dtypes: float64(2), int64(2), object(1)
memory usage: 19.0+ KB

   id                                           name   latitude   longitude  dpcapacity
0   2  (GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.  20.666378 -103.348820          15
1   3     (GDL-002) C. Colonias  / Av.  Niños héroes  20.667228 -103.366000          15
2   4          (GDL-003) C. Vidrio / Av. Chapultepec  20.667690 -103.368252          19
3   5           (GDL-004) C. Ghilardi /C. Miraflores  20.691847 -103.362549          19
4   6  (GDL-005) C. San Diego /Calzada Independencia  20.

Nomenclatura de estaciones. Aplicando la misma regla de lectura que con los viajes, el archivo se abre en UTF-8 al primer intento. Esto confirma que la decisión de no leer todo con Latin-1 era correcta: los nombres de las estaciones tienen acentos y ñ (por ejemplo, Niños héroes), que con Latin-1 habrían salido corruptos.

El archivo tiene 484 estaciones y 5 columnas, sin valores faltantes:

id: identificador numérico de la estación. Por su nombre y su tipo, todo indica que es el mismo identificador que origen_id y destino_id en los viajes; se verifica más adelante al cruzar ambos archivos.
name: nombre de la estación. Empieza con un código entre paréntesis (por ejemplo, GDL-001) seguido de las calles donde se ubica.
latitude y longitude: coordenadas de la estación.
dpcapacity: por el nombre, es probablemente la capacidad de la estación, es decir, el número de anclajes. Es la columna clave para P2, porque permite saber cuándo una estación estaría vacía o llena.

A diferencia de los viajes, los nombres de las columnas están en inglés. Al igual que con los viajes, se renombrarán a un esquema propio para que el cruce sea claro.

En los viajes, los identificadores de estación van del 2 al 503, y aquí hay 484 estaciones. Esto no indica todavía un problema (los identificadores no tienen por qué ser consecutivos), pero se revisará qué estaciones aparecen en un archivo y no en el otro.

In [28]:
print(f"Identificadores repetidos: {estaciones['id'].duplicated().sum()}")
print()
print(estaciones[["id", "latitude", "longitude", "dpcapacity"]].describe().round(4).to_string())
print()
print("Valores de dpcapacity (anclajes: número de estaciones):")
print(estaciones["dpcapacity"].value_counts().sort_index().to_string())

Identificadores repetidos: 0

             id  latitude  longitude  dpcapacity
count  484.0000  484.0000   484.0000    484.0000
mean   254.7583   20.6785  -103.3619     16.8822
std    148.2938    0.0220     0.0316      4.1269
min      2.0000   20.6361  -103.4332      0.0000
25%    129.7500   20.6651  -103.3842     15.0000
50%    252.0000   20.6744  -103.3624     15.0000
75%    379.2500   20.6886  -103.3393     19.0000
max    519.0000   20.7395  -103.2971     39.0000

Valores de dpcapacity (anclajes: número de estaciones):
dpcapacity
0       4
11     54
15    201
19    175
23     39
27      2
30      1
31      7
39      1


Rangos de la nomenclatura.

Identificadores: no hay repetidos, así que cada estación aparece una sola vez y el cruce con los viajes no duplicará filas. Van del 2 al 519, mientras que en los viajes las estaciones solo llegan hasta la 503. Por lo tanto, hay estaciones en el catálogo que nunca aparecen en los viajes, al menos las numeradas por encima de 503. Se revisará cuántas son al cruzar los dos archivos.
Coordenadas: la latitud va de 20.636 a 20.740 y la longitud de −103.433 a −103.297. Es un área de unos 11 × 14 km, coherente con una sola zona urbana (el área metropolitana de Guadalajara, donde opera MiBici, dicha información fue calculada con la ayuda de Claude de Anthropic). No hay ceros ni valores fuera de lugar.
Capacidad (dpcapacity): los valores son enteros y la mayoría están entre 11 y 39, lo que confirma que la columna representa el número de anclajes de cada estación. Las capacidades más comunes son 15 (201 estaciones) y 19 (175), que juntas suman el 78 % del catálogo.

Hallazgo importante para P2: hay 4 estaciones con capacidad 0. Una estación sin anclajes no puede guardar bicicletas, así que el riesgo de quedarse vacía o llena no está definido para ella: cualquier regla que compare el balance de bicicletas con la capacidad dividiría entre cero o marcaría la estación como llena todo el tiempo. Hay que revisar si estas estaciones tienen viajes registrados: si no tienen, se pueden excluir sin perder información; si los tienen, la capacidad registrada sería un error.

Ahora, crucemos nomenclatura con los viajes.

Paso 1. Renombrar las columnas de la nomenclatura

In [29]:
estaciones = estaciones.rename(columns={
    "id": "estacion_id",
    "name": "nombre",
    "latitude": "latitud",
    "longitude": "longitud",
    "dpcapacity": "capacidad",
})
print(estaciones.columns.tolist())

['estacion_id', 'nombre', 'latitud', 'longitud', 'capacidad']


Paso 2. ¿Qué estaciones están en un archivo y no en el otro?

Se usa la misma lógica de conjuntos que con las columnas al principio del notebook.

In [30]:
en_viajes = set(viajes_limpio["origen_id"]) | set(viajes_limpio["destino_id"])
en_catalogo = set(estaciones["estacion_id"])

print(f"Estaciones distintas en los viajes:      {len(en_viajes)}")
print(f"Estaciones en el catálogo:               {len(en_catalogo)}")
print(f"En ambos:                                {len(en_viajes & en_catalogo)}")
print(f"En los viajes pero NO en el catálogo:    {sorted(en_viajes - en_catalogo)}")
print(f"En el catálogo pero sin ningún viaje:    {len(en_catalogo - en_viajes)}")

Estaciones distintas en los viajes:      472
Estaciones en el catálogo:               484
En ambos:                                470
En los viajes pero NO en el catálogo:    [405, 407]
En el catálogo pero sin ningún viaje:    14


Cruce entre la nomenclatura y los viajes. Primero se renombran las columnas de la nomenclatura a un esquema propio en español (estacion_id, nombre, latitud, longitud y capacidad), igual que se hizo con los viajes.

Después se comparan las estaciones que aparecen en los viajes (como origen o como destino) con las del catálogo:

En los viajes aparecen 430 estaciones distintas, y 428 de ellas están en el catálogo. Esto confirma que estacion_id es el mismo identificador que origen_id y destino_id, y que los dos archivos se pueden cruzar.
2 estaciones de los viajes no están en el catálogo: la 405 y la 407. Para ellas no hay capacidad ni coordenadas, así que para P2 no es posible calcular su riesgo de vaciado o llenado.
56 estaciones del catálogo no tienen ningún viaje en los 18 meses analizados. Al no tener salidas ni llegadas, no aportan información para P2.

Quedan dos preguntas abiertas: cuántos viajes tienen las estaciones 405 y 407 y en qué periodo aparecen, y qué estaciones son las 56 sin viajes, en particular si entre ellas están las 4 con capacidad 0.

Ahora, Las estaciones 405 y 407: ¿cuántos viajes y cuándo?

In [31]:
for estacion in sorted(en_viajes - en_catalogo):
    viajes_estacion = viajes_limpio[(viajes_limpio["origen_id"] == estacion) | (viajes_limpio["destino_id"] == estacion)]
    print(f"Estación {estacion}: {len(viajes_estacion):,} viajes, "
          f"de {viajes_estacion['periodo'].min()} a {viajes_estacion['periodo'].max()}")

Estación 405: 3 viajes, de 2026_06 a 2026_06
Estación 407: 3,931 viajes, de 2026_06 a 2026_07


Las estaciones sin viajes y las de capacidad 0

In [32]:
sin_viajes = estaciones[~estaciones["estacion_id"].isin(en_viajes)]

print("Capacidad de las estaciones sin viajes:")
print(sin_viajes["capacidad"].value_counts().sort_index().to_string())

print("\nEstaciones con capacidad 0:")
capacidad_cero = estaciones[estaciones["capacidad"] == 0]
print(capacidad_cero.to_string())
print(f"\n¿Alguna tiene viajes? {capacidad_cero['estacion_id'].isin(en_viajes).any()}")

Capacidad de las estaciones sin viajes:
capacidad
0     4
11    2
15    6
19    2

Estaciones con capacidad 0:
     estacion_id                                           nombre    latitud    longitud  capacidad
94            99    ex-(ZPN-009) C. Del Corral / C. Nicolás Bravo  20.685108 -103.333746          0
104          109  ex-(ZPN-019) C. Cabecera Mpal. Juan Manuel / C.  20.723221 -103.385583          0
111          116  ex-(ZPN-026) C. 16 Sep. / C. Fray A. de Segovia  20.686920 -103.334670          0
129          138                                     ex-(ZPN-044)  20.697050 -103.363220          0

¿Alguna tiene viajes? False


Estaciones 405 y 407 (en los viajes, pero no en el catálogo). Las dos aparecen solo en junio de 2026, que forma parte del conjunto de prueba. Su peso es muy distinto: la 405 tiene apenas 3 viajes, pero la 407 tiene 1,962, así que no es un caso marginal. Como no están en el catálogo, no tienen capacidad ni coordenadas, y por lo tanto no se puede calcular su riesgo. Para P2 se excluyen como estaciones a predecir. Los viajes que las conectan con otras estaciones sí se conservan, porque las salidas y llegadas de esas otras estaciones son reales.

Estaciones con capacidad 0. Las 4 tienen un nombre que empieza con “ex-” (por ejemplo, ex-(ZPN-009)), lo que sugiere que son estaciones que ya no operan, conservadas en el catálogo con su código anterior. Ninguna tiene viajes en los 18 meses. Se excluyen sin perder información, y con eso se resuelve el problema de la capacidad 0 señalado antes.

Estaciones sin viajes. De las 56 estaciones del catálogo sin viajes, 4 son las de capacidad 0. Las otras 52 tienen capacidades normales (11, 15 o 19 anclajes). Como el catálogo es de septiembre de 2026, posterior al último mes de viajes (junio de 2026), una explicación posible es que sean estaciones abiertas entre julio y septiembre de 2026; con estos datos no se puede confirmar. En cualquier caso, al no tener salidas ni llegadas, no aportan información a P2 y se excluyen.

Que las estaciones 405 y 407 aparezcan solo en junio de 2026 plantea una pregunta más amplia: ¿hay estaciones que empiezan a operar a mitad del periodo? Si es así, el modelo no las vería en el entrenamiento y se encontraría con ellas por primera vez en validación o en prueba.

In [33]:
# Cada viaje genera dos "apariciones": una en la estación de origen y otra en la de destino
apariciones = pd.concat([
    viajes_limpio[["origen_id", "periodo"]].rename(columns={"origen_id": "estacion_id"}),
    viajes_limpio[["destino_id", "periodo"]].rename(columns={"destino_id": "estacion_id"}),
])

actividad = apariciones.groupby("estacion_id")["periodo"].agg(primer_mes="min", ultimo_mes="max")

print("Estaciones según el primer mes en que aparecen:")
print(actividad["primer_mes"].value_counts().sort_index().to_string())
print("\nEstaciones según el último mes en que aparecen:")
print(actividad["ultimo_mes"].value_counts().sort_index().to_string())
print("\nEstaciones activas por mes:")
print(apariciones.groupby("periodo")["estacion_id"].nunique().to_string())

Estaciones según el primer mes en que aparecen:
primer_mes
2025_01    368
2025_06      1
2025_10      1
2025_12      1
2026_05     12
2026_06     47
2026_07     27
2026_08     15

Estaciones según el último mes en que aparecen:
ultimo_mes
2025_06      1
2026_06      1
2026_07      1
2026_08      1
2026_09    468

Estaciones activas por mes:
periodo
2025_01    368
2025_02    367
2025_03    367
2025_04    367
2025_05    367
2025_06    368
2025_07    367
2025_08    367
2025_09    367
2025_10    368
2025_11    367
2025_12    368
2026_01    368
2026_02    368
2026_03    369
2026_04    368
2026_05    371
2026_06    427
2026_07    453
2026_08    467
2026_09    468


Estaciones a lo largo del tiempo. El número de estaciones activas es muy estable durante casi todo el periodo: entre 367 y 369 por mes de enero de 2025 a abril de 2026. En mayo de 2026 sube a 371 y en junio de 2026 salta a 427, es decir, 59 estaciones más que en abril.

Lo confirma el primer mes en que aparece cada estación: 368 ya estaban activas en enero de 2025, solo 3 aparecen después de forma aislada (en junio, octubre y diciembre de 2025) y 59 aparecen por primera vez en mayo (12) y junio (47) de 2026. Las estaciones 405 y 407, que no están en el catálogo, forman parte de este grupo. En sentido contrario, casi no hay estaciones que dejen de operar: solo 3 tienen su último viaje antes de junio de 2026.

Implicación para P2: las 59 estaciones nuevas caen todas en el conjunto de prueba (abril a junio de 2026). El modelo no las habrá visto ni en el entrenamiento ni en la validación, y no tienen historial para calcular variables basadas en el pasado (como los viajes de horas o días anteriores). Además, sus primeras semanas de operación probablemente no son representativas de su uso normal.

Las estaciones que aparecen por primera vez durante el periodo de prueba (mayo y junio de 2026) se excluyen de la evaluación de P2, porque el modelo no las vio durante el entrenamiento, no tienen historial para calcular variables basadas en el pasado y sus primeras semanas de operación probablemente no son representativas. Los viajes que las conectan con otras estaciones se conservan, porque las salidas y llegadas de esas otras estaciones son reales. 

Cada viaje genera dos movimientos:

una salida en la estación de origen, a la hora en que empieza;
una llegada en la estación de destino, a la hora en que termina.

Si contamos ambos por estación y por hora, obtenemos cuántas bicis salieron y cuántas llegaron. La diferencia es el balance:

balance = llegadas − salidas

Elegimos ese orden de la resta para que el signo se lea de forma natural: positivo significa que la estación ganó bicis (se está llenando) y negativo, que perdió bicis (se está vaciando).

In [34]:
# Salidas: por estación de origen y hora de inicio
salidas = (
    viajes_limpio
    .assign(hora=viajes_limpio["inicio"].dt.floor("h"))
    .groupby(["origen_id", "hora"]).size()
    .rename("salidas")
)
salidas.index.names = ["estacion_id", "hora"]

# Llegadas: por estación de destino y hora de fin
llegadas = (
    viajes_limpio
    .assign(hora=viajes_limpio["fin"].dt.floor("h"))
    .groupby(["destino_id", "hora"]).size()
    .rename("llegadas")
)
llegadas.index.names = ["estacion_id", "hora"]

# Unir ambas: donde una estación tuvo salidas pero no llegadas (o al revés), se pone 0
flujo = pd.concat([salidas, llegadas], axis=1).fillna(0).astype(int).reset_index()

# Verificación: cada viaje debe contar exactamente una salida y una llegada
print(f"Viajes:          {len(viajes_limpio):,}")
print(f"Total salidas:   {flujo['salidas'].sum():,}")
print(f"Total llegadas:  {flujo['llegadas'].sum():,}")
print(f"Rango de horas:  {flujo['hora'].min()}  a  {flujo['hora'].max()}")

Viajes:          7,872,964
Total salidas:   7,872,964
Total llegadas:  7,872,964
Rango de horas:  2025-01-01 00:00:00  a  2026-10-01 00:00:00


Tabla de flujo por estación y hora. Cada viaje se convierte en una salida (en su estación de origen, a la hora en que empieza) y una llegada (en su estación de destino, a la hora en que termina). Se define el balance como llegadas − salidas: positivo indica que la estación ganó bicicletas en esa hora, y negativo, que las perdió.

Verificación: el total de salidas y el total de llegadas coinciden exactamente con el número de viajes (6,606,296), así que la agregación no perdió ni duplicó movimientos.

Recortar al periodo de análisis

In [35]:
from proyecto_mibici.config import P2_FIN_PERIODO
fin_periodo = pd.Timestamp(P2_FIN_PERIODO)
fuera = flujo["hora"] > fin_periodo
print(f"Filas fuera del periodo: {fuera.sum()}  |  llegadas: {flujo.loc[fuera, 'llegadas'].sum()}  |  salidas: {flujo.loc[fuera, 'salidas'].sum()}")

flujo = flujo[~fuera].reset_index(drop=True)
flujo["balance"] = flujo["llegadas"] - flujo["salidas"]

print(f"\nFilas (estación-hora): {len(flujo):,}")
print(flujo[["salidas", "llegadas", "balance"]].describe().round(2).to_string())

Filas fuera del periodo: 545154  |  llegadas: 1266743  |  salidas: 1266668

Filas (estación-hora): 2,793,741
          salidas    llegadas     balance
count  2793741.00  2793741.00  2793741.00
mean         2.36        2.36       -0.00
std          2.94        3.08        2.48
min          0.00        0.00      -88.00
25%          1.00        1.00       -1.00
50%          1.00        1.00        0.00
75%          3.00        3.00        1.00
max        207.00      211.00      100.00


Se excluyen las 55 filas del 1 de julio de 2026. Contienen 75 llegadas y ninguna salida, lo que confirma que son las llegadas de los 75 viajes que salieron el 30 de junio, y que esas horas están incompletas.

2,793,741 combinaciones de estación y hora. En una estación-hora típica hay pocas salidas y llegadas (mediana de 1 y percentil 75 de 3). El balance promedio también es 0, lo cual es esperable: en todo el sistema, cada bicicleta que sale de una estación llega a otra.

Ahora,

Valores extremos: hay estaciones con hasta 207 salidas o 211 llegadas en una sola hora, y balances de −88 y +100. Comparado con la capacidad de las estaciones (la mayoría tiene 15 o 19 anclajes), esto es muy alto. Podría explicarse por bicicletas que se sacan y se devuelven continuamente en la misma estación, o por algún problema en los registros.

Horas sin movimiento: la tabla solo contiene las horas con al menos un viaje; las horas sin salidas ni llegadas no aparecen. Por eso las estadísticas anteriores describen solo las horas con actividad. Para P2 esto importa porque una hora sin movimiento también es información (la estación no ganó ni perdió bicicletas), y para calcular el inventario acumulado o usar las horas anteriores como variables se necesita una serie completa, con todas las horas de cada estación.

Vamos a revisar los casos extremos: ¿son errores o algo real?

In [36]:
extremos = (
    flujo.nlargest(10, "salidas")
    .merge(estaciones[["estacion_id", "capacidad"]], on="estacion_id", how="left")
)
print(extremos.to_string(index=False))

 estacion_id                hora  salidas  llegadas  balance  capacidad
         194 2025-09-20 21:00:00      207       211        4         19
         194 2025-09-20 20:00:00      178       181        3         19
         194 2026-04-04 20:00:00      149       153        4         19
         194 2026-04-04 21:00:00      129       128       -1         19
         194 2026-02-07 20:00:00      121       113       -8         19
         194 2026-02-07 21:00:00      121       115       -6         19
         194 2025-11-01 21:00:00      109       102       -7         19
         194 2026-03-07 20:00:00      109        93      -16         19
         194 2026-05-02 20:00:00      105       107        2         19
          50 2025-09-20 20:00:00      104       114       10         23


¿Cómo son los viajes del caso más extremo?

In [37]:
caso = extremos.iloc[0]
viajes_caso = viajes_limpio[
    (viajes_limpio["origen_id"] == caso["estacion_id"])
    & (viajes_limpio["inicio"].dt.floor("h") == caso["hora"])
]

print(f"Viajes: {len(viajes_caso)}  |  usuarios distintos: {viajes_caso['usuario_id'].nunique()}")
print(f"Proporción con el mismo origen y destino: {viajes_caso['misma_estacion'].mean():.1%}")
print("\nDuración (min):")
print(viajes_caso["duracion_min"].describe().round(1).to_string())
print("\nDestinos más frecuentes:")
print(viajes_caso["destino_id"].value_counts().head().to_string())

Viajes: 207  |  usuarios distintos: 198
Proporción con el mismo origen y destino: 7.2%

Duración (min):
count    207.0
mean      22.3
std       11.0
min        0.4
25%       15.4
50%       22.5
75%       27.8
max       72.9

Destinos más frecuentes:
destino_id
51     26
50     21
194    15
52     13
49     12


In [38]:
print(estaciones[estaciones["estacion_id"].isin([194, 49, 50, 51, 52])].to_string(index=False))
print()
print(extremos["hora"].dt.day_name().value_counts().to_string())

 estacion_id                                        nombre   latitud    longitud  capacidad
          49        (GDL-047) C. Robles Gil / Av. Vallarta 20.674931 -103.362351         15
          50 (GDL-048) C. Constancio Hernández/ Av. Juaréz 20.674721 -103.358548         23
          51       (GDL-049) Lopez Cotilla / Penitenciaría 20.674140 -103.355535         39
          52         (GDL-050) C.Pedro Moreno /Federalismo 20.675746 -103.354579         23
         194     (GDL-127) Av. Vallarta / Av. de Los Arcos 20.674380 -103.384880         19

hora
Saturday    10


Casos extremos de actividad. 9 de las 10 estaciones-hora con más salidas son de la estación 194, y el décimo es de la estación 50.

No parecen errores del sistema. En el caso más extremo, los 207 viajes los hicieron 198 usuarios distintos, con una duración típica (mediana de 22.5 minutos), y solo el 7.2 % volvió a la misma estación. Son viajes reales y se conservan.

¿Cómo pueden salir 207 bicicletas de una estación de 19 anclajes en una hora? Porque en esa misma hora llegaron 211. Las bicicletas que llegan liberan anclajes y quedan disponibles para los siguientes usuarios, así que una estación puede tener muchas más salidas que anclajes si su rotación es alta. El balance de esa hora fue de solo +4.

Implicación para P2: el número de salidas por sí solo no indica riesgo; lo que importa es el balance acumulado respecto a la capacidad.

In [39]:
# Estaciones con capacidad válida en el catálogo
con_capacidad = set(estaciones.loc[estaciones["capacidad"] > 0, "estacion_id"])

# Estaciones que aparecen por primera vez durante el periodo de prueba (abril a junio de 2026)
nuevas_en_prueba = set(actividad.index[actividad["primer_mes"] >= "2026_04"])

# Estaciones que se pueden modelar: tienen viajes, están en el catálogo y tienen capacidad
modelables = en_viajes & en_catalogo & con_capacidad

print(f"Estaciones con viajes:                          {len(en_viajes)}")
print(f"  ...y en el catálogo con capacidad > 0:        {len(modelables)}")
print(f"Estaciones nuevas en el periodo de prueba:      {len(nuevas_en_prueba)}  (de ellas, en el catálogo: {len(nuevas_en_prueba & en_catalogo)})")
print(f"Modelables y no nuevas (se usan en la evaluación): {len(modelables - nuevas_en_prueba)}")

# Quitar de flujo las estaciones que no se pueden modelar (405 y 407)
filas_antes = len(flujo)
flujo = flujo[flujo["estacion_id"].isin(modelables)].reset_index(drop=True)
print(f"\nFilas de flujo eliminadas: {filas_antes - len(flujo)}")

# Marcar las estaciones nuevas, para excluirlas solo al evaluar
flujo["estacion_nueva"] = flujo["estacion_id"].isin(nuevas_en_prueba)

Estaciones con viajes:                          472
  ...y en el catálogo con capacidad > 0:        470
Estaciones nuevas en el periodo de prueba:      101  (de ellas, en el catálogo: 99)
Modelables y no nuevas (se usan en la evaluación): 371

Filas de flujo eliminadas: 327


Aplicación de las exclusiones. Se aplican sobre la tabla por estación, no sobre los viajes, para que las salidas y llegadas de las demás estaciones se conserven:

Se eliminan de flujo las estaciones que no se pueden modelar: la 405 y la 407 (sin capacidad en el catálogo). Las 4 estaciones de capacidad 0 y las 52 sin viajes no tenían filas en la tabla. Quedan 428 estaciones.
Las 57 estaciones nuevas del periodo de prueba que sí están en el catálogo se conservan en la tabla, pero se marcan con estacion_nueva, para excluirlas al evaluar. Quedan 371 estaciones para la evaluación.

Ahora, sigamos trabajando con las horas sin movimiento.

In [40]:
inicio_periodo = pd.Timestamp("2025-01-01 00:00")
todas_las_horas = pd.date_range(inicio_periodo, fin_periodo, freq="h")
print(f"Horas en el periodo: {len(todas_las_horas):,}")

# Primera y última hora con movimiento de cada estación
rango_estacion = flujo.groupby("estacion_id")["hora"].agg(primera_hora="min", ultima_hora="max")

# Todas las combinaciones de estación y hora
indice_completo = pd.MultiIndex.from_product(
    [sorted(flujo["estacion_id"].unique()), todas_las_horas],
    names=["estacion_id", "hora"],
)

serie = (
    flujo.set_index(["estacion_id", "hora"])[["salidas", "llegadas"]]
    .reindex(indice_completo, fill_value=0)
    .reset_index()
)

# Conservar solo las horas dentro del rango de actividad de cada estación
serie = serie.merge(rango_estacion, on="estacion_id")
dentro = (serie["hora"] >= serie["primera_hora"]) & (serie["hora"] <= serie["ultima_hora"])
serie = serie[dentro].drop(columns=["primera_hora", "ultima_hora"]).reset_index(drop=True)

serie["balance"] = serie["llegadas"] - serie["salidas"]
serie["estacion_nueva"] = serie["estacion_id"].isin(nuevas_en_prueba)

# Verificaciones
print(f"Filas (estación-hora): {len(serie):,}")
print(f"Salidas:  serie {serie['salidas'].sum():,}  |  flujo {flujo['salidas'].sum():,}")
print(f"Llegadas: serie {serie['llegadas'].sum():,}  |  flujo {flujo['llegadas'].sum():,}")
sin_movimiento = ((serie["salidas"] == 0) & (serie["llegadas"] == 0)).mean()
print(f"Horas sin ningún movimiento: {sin_movimiento:.1%}")
print()
print(serie[["salidas", "llegadas", "balance"]].describe().round(2).to_string())

Horas en el periodo: 13,104
Filas (estación-hora): 4,838,538
Salidas:  serie 6,605,326  |  flujo 6,605,326
Llegadas: serie 6,605,152  |  flujo 6,605,152
Horas sin ningún movimiento: 42.3%

          salidas    llegadas     balance
count  4838538.00  4838538.00  4838538.00
mean         1.37        1.37       -0.00
std          2.52        2.61        1.89
min          0.00        0.00      -88.00
25%          0.00        0.00        0.00
50%          0.00        0.00        0.00
75%          2.00        2.00        0.00
max        207.00      211.00      100.00


Serie completa por estación y hora. La tabla flujo solo tenía las horas con al menos un movimiento. Para poder acumular el balance y usar las horas anteriores como información, se completa una serie continua para cada estación, con todas las horas del periodo (13,104 horas, del 1 de enero de 2025 al 30 de junio de 2026), rellenando con 0 salidas y 0 llegadas las horas sin viajes.

Cada estación se completa solo entre su primera y su última hora con movimiento. Así se evita que una estación nueva tenga meses de horas “sin uso” antes de existir, o que una estación que dejó de operar las tenga después. Por eso la tabla tiene 4,838,538 filas y no las 5.6 millones que resultarían de multiplicar 428 estaciones por 13,104 horas.

Verificación: las salidas y llegadas totales de la serie coinciden exactamente con las de flujo, así que completar las horas no agregó ni quitó movimientos. Ya no coinciden con el total de viajes porque se excluyeron las estaciones 405 y 407: los viajes que las conectan con otras estaciones solo cuentan en el extremo de la estación que sí se modela.

Resultado: el 42.3 % de las horas no tiene ningún movimiento. Esto cambia la imagen de una hora típica: antes, considerando solo las horas con actividad, la mediana era de 1 salida y 1 llegada; con la serie completa, la mediana es 0 y el promedio baja de 2.36 a 1.37. Para P2, esto significa que la mayor parte del tiempo una estación no cambia su número de bicicletas, y que el riesgo se concentra en las horas de mucha actividad.

Acumular el balance durante todo el periodo

In [41]:
serie = serie.sort_values(["estacion_id", "hora"]).reset_index(drop=True)
serie["balance_acumulado"] = serie.groupby("estacion_id")["balance"].cumsum()

resumen_acumulado = (
    serie.groupby("estacion_id")["balance_acumulado"]
    .agg(minimo="min", maximo="max", final="last")
    .join(estaciones.set_index("estacion_id")["capacidad"])
)
resumen_acumulado["rango"] = resumen_acumulado["maximo"] - resumen_acumulado["minimo"]

print(resumen_acumulado[["final", "rango", "capacidad"]].describe().round(1).to_string())
print(f"\nEstaciones cuyo rango supera su capacidad: "
      f"{(resumen_acumulado['rango'] > resumen_acumulado['capacidad']).sum()} de {len(resumen_acumulado)}")
print("\nEstaciones con el acumulado final más bajo y más alto:")
print(pd.concat([resumen_acumulado.nsmallest(3, "final"), resumen_acumulado.nlargest(3, "final")]).to_string())

         final    rango  capacidad
count    428.0    428.0      428.0
mean      -0.4   2088.9       17.0
std     3798.7   3181.0        4.0
min   -34081.0      0.0       11.0
25%    -1571.2    505.5       15.0
50%     -223.5   1339.0       15.0
75%      860.5   2480.5       19.0
max    32998.0  34081.0       39.0

Estaciones cuyo rango supera su capacidad: 395 de 428

Estaciones con el acumulado final más bajo y más alto:
             minimo  maximo  final  capacidad  rango
estacion_id                                         
194          -34081       0 -34081         19  34081
271           -9686       2  -9685         31   9688
288           -7346       0  -7346         31   7346
51              -26   33003  32998         39  33029
35               -3   21435  21435         15  21438
260              -1   15176  15176         15  15177


¿Se puede reconstruir el inventario acumulando el balance? Si los viajes registraran todos los movimientos de bicicletas, el balance acumulado de cada estación debería mantenerse dentro de un rango compatible con su capacidad: una estación no puede tener más bicicletas que anclajes, ni menos de cero.

El resultado muestra lo contrario. 395 de las 428 estaciones tienen un rango de balance acumulado mayor que su capacidad, y en muchos casos por mucho: la mediana del rango es de 1,339 bicicletas, cuando la capacidad típica es de 15. Los casos extremos son imposibles en sentido literal: la estación 194, con 19 anclajes, acumula −34,081 bicicletas (pierde muchas más de las que recibe), y la estación 51, con 39 anclajes, acumula +32,998 (recibe muchas más de las que salen).

Interpretación: hay movimientos de bicicletas que no aparecen en los datos de viajes. La explicación más probable es el rebalanceo: el operador traslada bicicletas en vehículos, desde las estaciones que se llenan hacia las que se vacían, para mantener el servicio. Precisamente ese es el trabajo del equipo de rebalanceo al que va dirigida P2. También podrían influir las bicicletas que se retiran para mantenimiento. Con estos datos no se puede distinguir entre ambas causas, pero el efecto es el mismo: el balance acumulado se aleja sin límite del inventario real.

Implicación para P2: no es posible reconstruir el inventario real de una estación acumulando su balance desde el inicio del periodo. Cualquier aproximación tiene que limitar la ventana de acumulación, suponiendo que el rebalanceo devuelve periódicamente las estaciones a un estado “normal”. Se evalúa a continuación una ventana diaria.

hay que decidir a qué hora empieza un “día”

In [42]:
actividad_hora = serie.groupby(serie["hora"].dt.hour)[["salidas", "llegadas"]].sum()
actividad_hora["porcentaje"] = (
    (actividad_hora["salidas"] + actividad_hora["llegadas"])
    / (actividad_hora["salidas"] + actividad_hora["llegadas"]).sum() * 100
).round(2)
print(actividad_hora.to_string())

      salidas  llegadas  porcentaje
hora                               
0       41451     46459        0.67
1          58      6808        0.05
2           2        68        0.00
3           0        15        0.00
4           3         5        0.00
5       55092     41778        0.73
6      188109    158969        2.63
7      369372    333590        5.32
8      511077    491525        7.59
9      365471    398316        5.78
10     320614    320942        4.86
11     322475    322456        4.88
12     343331    338397        5.16
13     386576    384555        5.84
14     432757    430797        6.54
15     439829    441879        6.67
16     417622    416627        6.32
17     491319    469785        7.28
18     606047    598718        9.12
19     460840    489103        7.19
20     347690    366504        5.41
21     268394    283327        4.18
22     156169    174860        2.51
23      81028     89669        1.29


¿A qué hora empieza un día? La actividad por hora del día muestra que el sistema prácticamente no opera de madrugada: entre la 1:00 y las 5:00 casi no hay salidas (58 a la 1:00 y, después, 3 o menos por hora en 18 meses). Las pocas llegadas de la 1:00 corresponden a viajes que salieron antes de medianoche. A medianoche todavía hay actividad (0.67 % de los movimientos), mientras que a las 3:00 no hay ninguna salida. Por eso se define un día operativo que empieza a las 3:00: el corte cae en un momento sin movimientos y no parte en dos ningún periodo de actividad.

In [43]:
from proyecto_mibici.config import P2_HORA_INICIO_DIA
serie["dia_operativo"] = (serie["hora"] - pd.Timedelta(hours=P2_HORA_INICIO_DIA)).dt.floor("D")

serie["balance_acumulado_dia"] = serie.groupby(["estacion_id", "dia_operativo"])["balance"].cumsum()

rango_diario = (
    serie.groupby(["estacion_id", "dia_operativo"])["balance_acumulado_dia"]
    .agg(minimo="min", maximo="max")
    .reset_index()
    .merge(estaciones[["estacion_id", "capacidad"]], on="estacion_id")
)
rango_diario["rango"] = rango_diario["maximo"] - rango_diario["minimo"]
rango_diario["rango_sobre_capacidad"] = rango_diario["rango"] / rango_diario["capacidad"]

supera = rango_diario["rango"] > rango_diario["capacidad"]
print(f"Estaciones-día: {len(rango_diario):,}")
print(f"Estaciones-día cuyo rango supera la capacidad: {supera.sum():,} ({supera.mean():.1%})")
print()
print(rango_diario[["rango", "rango_sobre_capacidad"]]
      .describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(2).to_string())

Estaciones-día: 201,846
Estaciones-día cuyo rango supera la capacidad: 24,460 (12.1%)

           rango  rango_sobre_capacidad
count  201846.00              201846.00
mean       10.32                   0.61
std        10.85                   0.51
min         0.00                   0.00
50%         8.00                   0.48
75%        13.00                   0.79
90%        19.00                   1.09
95%        24.00                   1.36
99%        41.00                   2.27
max       247.00                   9.21


/var/folders/fx/g8j27htd41lck7mgmk79vccw0000gn/T/ipykernel_29873/4103163804.py:2: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  serie["dia_operativo"] = (serie["hora"] - pd.Timedelta(hours=P2_HORA_INICIO_DIA)).dt.floor("D")


Balance acumulado por día operativo. Reiniciando el acumulado al inicio de cada día operativo, el resultado cambia radicalmente respecto al acumulado de todo el periodo (donde 395 de 428 estaciones superaban su capacidad):

Solo el 12.1 % de las 201,846 estaciones-día tiene un rango de balance mayor que la capacidad de la estación.
En un día típico, el movimiento neto de una estación ocupa menos de la mitad de su capacidad (mediana de rango_sobre_capacidad = 0.48). El percentil 75 es 0.79, y la capacidad solo se supera a partir del percentil 90 (1.09).

Con una ventana diaria, el balance acumulado es compatible con la capacidad de las estaciones en la gran mayoría de los días. Esto hace plausible el supuesto de que cada día operativo empieza en un estado “normal”, restablecido por movimientos que no aparecen en los datos (como el rebalanceo), pero no lo demuestra: los datos no muestran qué ocurre fuera de los viajes. En el 12.1 % de los días que no caben, es probable que haya rebalanceo durante el día, sobre todo en estaciones de alta rotación como la 194.

Implicación para P2: el balance acumulado dentro del día operativo es una aproximación razonable de cuántas bicicletas ganó o perdió una estación desde el inicio del día. Lo que sigue sin conocerse es cuántas tenía al empezar, y eso es lo que habrá que suponer para definir cuándo una estación está en riesgo de quedarse vacía o llena.

In [44]:
from proyecto_mibici.config import P2_UMBRAL
UMBRAL = P2_UMBRAL  # fracción de la capacidad: la estación empieza el día con la mitad llena

serie = serie.merge(estaciones[["estacion_id", "capacidad"]], on="estacion_id", how="left")

serie["riesgo_vacia"] = serie["balance_acumulado_dia"] <= -UMBRAL * serie["capacidad"]
serie["riesgo_llena"] = serie["balance_acumulado_dia"] >= UMBRAL * serie["capacidad"]
serie["en_riesgo"] = serie["riesgo_vacia"] | serie["riesgo_llena"]

print(f"Estaciones-hora: {len(serie):,}")
print(f"Sin capacidad asignada: {serie['capacidad'].isna().sum()}")
print(f"Riesgo de quedarse vacía:   {serie['riesgo_vacia'].mean():.2%}")
print(f"Riesgo de llenarse:         {serie['riesgo_llena'].mean():.2%}")
print(f"En riesgo (cualquiera):     {serie['en_riesgo'].mean():.2%}")
print(f"Ambos a la vez:             {(serie['riesgo_vacia'] & serie['riesgo_llena']).sum()}")

Estaciones-hora: 4,838,538
Sin capacidad asignada: 0
Riesgo de quedarse vacía:   6.13%
Riesgo de llenarse:         6.12%
En riesgo (cualquiera):     12.26%
Ambos a la vez:             0


Variable objetivo de P2. Como los datos no incluyen el inventario de las estaciones, se aproxima con el balance acumulado del día operativo y se establece un supuesto sobre el punto de partida: cada estación empieza el día operativo con la mitad de su capacidad. Con ese supuesto, se definen dos tipos de riesgo:

Riesgo de quedarse vacía: el balance acumulado del día es menor o igual a −50 % de la capacidad, es decir, la estación ya perdió las bicicletas con las que empezó.
Riesgo de llenarse: el balance acumulado es mayor o igual a +50 % de la capacidad, es decir, ya no le quedan anclajes libres.

La variable en_riesgo vale verdadero si ocurre cualquiera de los dos. El umbral de 50 % se justifica con el análisis anterior: en un día típico, el movimiento neto de una estación ocupa menos de la mitad de su capacidad (mediana de 0.48), así que solo los días con un desbalance fuera de lo común llegan al umbral.

Resultado: el 12.26 % de las estaciones-hora están en riesgo, repartidas casi por igual entre riesgo de vaciarse (6.13 %) y de llenarse (6.12 %). Ninguna hora tiene los dos riesgos a la vez, lo cual es coherente con la definición. Las clases están desbalanceadas (aproximadamente 1 hora en riesgo por cada 7 sin riesgo), lo que habrá que considerar al elegir las métricas del modelo.

Limitación: el supuesto de partida es una simplificación. En la realidad, cada estación empieza el día con un número distinto de bicicletas, y el rebalanceo durante el día modifica el inventario sin quedar registrado.

In [45]:
from proyecto_mibici.config import P2_INICIO_VALIDACION, P2_INICIO_PRUEBA
serie["conjunto"] = np.select(
    [serie["hora"] < P2_INICIO_VALIDACION, serie["hora"] < P2_INICIO_PRUEBA],
    ["entrenamiento", "validacion"],
    default="prueba",
)
orden = ["entrenamiento", "validacion", "prueba"]

print("Todas las estaciones:")
print(serie.groupby("conjunto").agg(
    filas=("en_riesgo", "size"),
    estaciones=("estacion_id", "nunique"),
    en_riesgo=("en_riesgo", "mean"),
    riesgo_vacia=("riesgo_vacia", "mean"),
    riesgo_llena=("riesgo_llena", "mean"),
).reindex(orden).round(4).to_string())

print("\nSin las estaciones nuevas (como se evaluará):")
print(serie[~serie["estacion_nueva"]].groupby("conjunto").agg(
    filas=("en_riesgo", "size"),
    estaciones=("estacion_id", "nunique"),
    en_riesgo=("en_riesgo", "mean"),
).reindex(orden).round(4).to_string())

Todas las estaciones:
                 filas  estaciones  en_riesgo  riesgo_vacia  riesgo_llena
conjunto                                                                 
entrenamiento  3215126         371     0.1313        0.0681        0.0632
validacion      797040         369     0.1090        0.0507        0.0583
prueba          826372         426     0.1016        0.0451        0.0564

Sin las estaciones nuevas (como se evaluará):
                 filas  estaciones  en_riesgo
conjunto                                     
entrenamiento  3215126         371     0.1313
validacion      797040         369     0.1090
prueba          803562         369     0.1037


Frecuencia del riesgo por conjunto. Se asigna cada estación-hora a su conjunto según la partición temporal. La proporción de horas en riesgo disminuye con el tiempo: 13.13 % en entrenamiento, 10.90 % en validación y 10.16 % en prueba. La disminución ocurre en los dos tipos de riesgo, pero es más marcada en el riesgo de quedarse vacía (de 6.81 % a 4.51 %) que en el de llenarse (de 6.32 % a 5.64 %).

Al excluir las estaciones nuevas, como se hará en la evaluación, el conjunto de prueba pasa de 426 a 369 estaciones y su proporción de riesgo apenas cambia (10.37 %). Por lo tanto, la disminución no se debe a las estaciones nuevas.

Implicaciones para el modelo:

Las clases están desbalanceadas en los tres conjuntos: alrededor de 1 de cada 10 estaciones-hora está en riesgo. Una métrica como la accuracy sería engañosa, porque un modelo que nunca predijera riesgo acertaría casi el 90 % de las veces. Se usarán métricas centradas en la clase positiva, como F1, recall y PR-AUC.
La proporción de riesgo cambia entre conjuntos, así que el umbral de decisión del modelo debe ajustarse en validación, que se parece más a la prueba que el entrenamiento.
Con estos datos no se puede saber la causa de la disminución. Una hipótesis es que haya cambiado la forma de operar o de rebalancear el sistema en 2026; otra, que el uso de las estaciones se haya vuelto más equilibrado.

Patrones del riesgo por hora del día y día de la semana

In [46]:
entrenamiento = serie[serie["conjunto"] == "entrenamiento"]
columnas_riesgo = ["en_riesgo", "riesgo_vacia", "riesgo_llena"]

print("Porcentaje de horas en riesgo, por hora del día (entrenamiento):")
print(entrenamiento.groupby(entrenamiento["hora"].dt.hour)[columnas_riesgo]
      .mean().mul(100).round(1).to_string())

por_dia = entrenamiento.groupby(entrenamiento["hora"].dt.dayofweek)[columnas_riesgo].mean().mul(100).round(1)
por_dia.index = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
print("\nPorcentaje de horas en riesgo, por día de la semana (entrenamiento):")
print(por_dia.to_string())

Porcentaje de horas en riesgo, por hora del día (entrenamiento):
      en_riesgo  riesgo_vacia  riesgo_llena
hora                                       
0          20.5          10.8           9.7
1          20.5          10.7           9.8
2          20.5          10.7           9.8
3           0.0           0.0           0.0
4           0.0           0.0           0.0
5           0.0           0.0           0.0
6           0.2           0.1           0.1
7           2.8           1.6           1.2
8           8.4           4.5           3.9
9          12.2           5.6           6.6
10         13.8           6.4           7.4
11         14.7           6.9           7.8
12         15.4           7.3           8.0
13         15.6           7.6           8.1
14         15.0           7.4           7.6
15         14.3           7.3           7.0
16         13.8           7.2           6.6
17         14.3           7.8           6.5
18         16.2           9.2           7.1
19         

Patrones del riesgo por hora y día (solo entrenamiento). Se usa únicamente el conjunto de entrenamiento para que las decisiones sobre las variables del modelo no utilicen información de validación ni de prueba.

Por hora del día: el riesgo es 0 % entre las 3:00 y las 5:00 y aumenta a lo largo del día: 8.4 % a las 8:00, alrededor de 15 % al mediodía y 20.3 % a las 23:00, y se mantiene en 20.5 % hasta las 2:00. Este patrón es en buena parte consecuencia de cómo se definió la variable objetivo: el acumulado se reinicia a las 3:00, así que a esa hora ninguna estación puede estar en riesgo, y a medida que avanza el día el balance tiene más tiempo para alejarse de cero. A partir de las 17:00, el riesgo de quedarse vacía crece más rápido que el de llenarse (10.8 % contra 9.6 % a las 23:00).

Por día de la semana: el riesgo es mayor de martes a jueves (alrededor de 16 %) y mucho menor en fin de semana (9.2 % el sábado y 6.2 % el domingo). El lunes (12.8 %) queda en un punto intermedio. Una posible explicación es que el uso entre semana sea más direccional (por ejemplo, por traslados al trabajo o la escuela), pero estos datos no lo permiten confirmar.

Implicaciones para las variables del modelo:

La hora del día y el día de la semana son variables claramente relevantes, y deben incluirse.
Como el riesgo de una hora depende del acumulado del día, es probable que el estado actual de la estación (su balance acumulado hasta ahora y si ya está en riesgo) sea el predictor más fuerte de lo que pasará en la siguiente hora. Esto sugiere un baseline de persistencia: predecir que la estación estará en la siguiente hora en el mismo estado que en la actual.

Resumen del EDA y decisiones para P2

Lectura de los datos

Hay 18 archivos trimestrales (2025 T1 a 2026 T2). 4 de ellos no se pueden leer en UTF-8: fallan en el byte 0xf1, que en Latin-1 es ñ. Decisión: leer en UTF-8 y, si falla, usar Latin-1.
Los encabezados no son iguales en todos los archivos: aparecen 9 nombres de columna distintos y solo 7 son comunes. Decisión: renombrar las columnas a un esquema único en ASCII (viaje_id, usuario_id, genero, anio_nacimiento, inicio, fin, origen_id, destino_id, periodo).
Los valores faltantes vienen como texto: NULL en género y NA en año de nacimiento. Decisión: tratarlos como nulos al leer.
Con todo esto quedan 6,622,260 viajes leídos.

Calidad de los viajes

Género y año de nacimiento tienen nulos y valores imposibles (por ejemplo, un año mínimo de 199). Decisión: no se usan en P2, que se trabaja a nivel estación-hora y no a nivel usuario. Por eso no se limpian.
Hay 27 filas que son copias de otro viaje (idénticas salvo el id).
Hay 15,954 viajes de menos de 10 segundos entre estaciones distintas, que no son físicamente posibles.
Decisión: eliminar ambos grupos. En total se eliminan 15,964 filas y quedan 6,606,296 viajes.
Ningún viaje empieza fuera del mes de su archivo. Sí hay 481 viajes que terminan al mes siguiente, lo cual es consistente con viajes que cruzan la medianoche del fin de mes.

Estaciones

El catálogo de estaciones (septiembre de 2026) tiene 484 estaciones; 4 de ellas tienen capacidad 0.
Las estaciones 405 y 407 aparecen en los viajes pero no en el catálogo. Decisión: excluirlas, porque sin capacidad no se puede definir el riesgo.
59 estaciones aparecen por primera vez en mayo o junio de 2026 (57 de ellas están en el catálogo). Decisión: conservarlas en la serie con la marca estacion_nueva, pero excluirlas al evaluar, porque el modelo nunca las vio durante el entrenamiento.
Supuesto: la capacidad proviene de un catálogo de septiembre de 2026. Se asume que no cambió durante el periodo de estudio, pero no lo podemos verificar.

Serie estación-hora

Por cada estación y hora se cuentan las salidas, las llegadas y el balance (llegadas − salidas). La serie se completa con ceros desde la primera hasta la última hora activa de cada estación. El resultado tiene 4,838,538 filas; el 42.3 % de las horas no tiene movimientos.
Las llegadas posteriores al 30 de junio de 2026 a las 23:00 se excluyen.
Si se acumula el balance sobre todo el periodo, 395 de 428 estaciones superan su capacidad. Hipótesis: el sistema rebalancea bicicletas y ese movimiento no aparece en los datos. Por eso el acumulado de largo plazo no sirve como inventario.
Decisión: usar un día operativo que empieza a las 3:00, que es la hora de menor actividad, y reiniciar el balance acumulado al inicio de cada día.

Variable objetivo

Regla: una estación está en_riesgo cuando su balance acumulado del día operativo es menor o igual a −50 % de su capacidad (riesgo_vacia) o mayor o igual a +50 % de su capacidad (riesgo_llena). Con esta regla, el 12.26 % de las horas está en riesgo.
Es una aproximación: no conocemos el inventario real de bicicletas, solo el cambio neto de cada día.

Partición temporal

Entrenamiento: 2025. Validación: enero a marzo de 2026. Prueba: abril a junio de 2026.
Las tasas de riesgo son 13.13 % en entrenamiento, 10.90 % en validación y 10.16 % en prueba. Si se excluyen las estaciones nuevas, la tasa de prueba es 10.37 %.
Decisión: como las clases están desbalanceadas, usar F1, recall y PR-AUC en lugar de exactitud, y ajustar el umbral de decisión con el conjunto de validación.

Variables sugeridas para el modelo

La hora del día, el día de la semana y el estado actual de la estación (su balance acumulado y si ya está en riesgo).
Baseline de persistencia: predecir que la estación estará en la siguiente hora en el mismo estado que en la actual.

### 6. Modelado Predictivo de Riesgo de Desbalance (Clasificación)

**Formulación del problema:**
El objetivo es predecir si una estación entrará en estado de desbalance (`en_riesgo`, ya sea por vaciado $\le -50\%$ o saturación $\ge +50\%$ de su capacidad) en la hora siguiente ($t+1$). Esta anticipación de una hora otorga una ventana operativa útil para que las cuadrillas de redistribución en vehículos programen rutas antes de que el usuario experimente desabasto o falta de anclajes.

**Estrategia de Modelado y Comparación:**
1. **Línea base (Baseline ingenuo de persistencia):** Asume inercia operativa inmediata; es decir, predice que el estado de la estación en $t+1$ será idéntico a su estado observado en $t$. Funciona como el umbral mínimo que cualquier algoritmo estadístico debe superar.
2. **Modelo candidato (Regresión Logística balanceada):** Se entrena sobre las variables temporales (`hora_del_dia`, `dia_semana`, `fin_de_semana`) y de flujo relativo (`acumulado_relativo`, `balance_ult_3h_relativo`). Debido a que la clase positiva representa únicamente $\sim 10\%$ de las observaciones, se configura `class_weight='balanced'` para penalizar desproporcionadamente los falsos negativos y ajustar la frontera de decisión ante el desbalance.

**Justificación de Métricas:**
Dado el marcado desbalance de clases ($1:7$ y hasta $1:9$ en periodos de baja actividad), la métrica tradicional de exactitud (*accuracy*) resulta engañosa (un modelo trivial que prediga siempre clase 0 obtendría $\sim 90\%$ de exactitud sin valor operativo). Se priorizan métricas enfocadas en la detección de la clase minoritaria: **Recall** (cobertura de estaciones en riesgo real), **Precision** (certeza al emitir una alerta para evitar viajes en vano de las cuadrillas), **F1-Score** y el área bajo la curva Precision-Recall (**PR-AUC**).

In [48]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from proyecto_mibici.config import P2_VARIABLES, RANDOM_SEED

# 1. Crear variables predictoras y rezago de objetivo a t+1
por_estacion = serie.groupby("estacion_id")
serie["hora_del_dia"] = serie["hora"].dt.hour
serie["dia_semana"] = serie["hora"].dt.dayofweek
serie["fin_de_semana"] = (serie["dia_semana"] >= 5).astype(int)
serie["acumulado_relativo"] = serie["balance_acumulado_dia"] / serie["capacidad"]
serie["balance_ult_3h_relativo"] = (
    por_estacion["balance"].rolling(3, min_periods=1).sum().reset_index(level=0, drop=True)
    / serie["capacidad"]
)
serie["target_en_riesgo"] = por_estacion["en_riesgo"].shift(-1)
df_mod = serie.dropna(subset=["target_en_riesgo"]).reset_index(drop=True)
df_mod["target_en_riesgo"] = df_mod["target_en_riesgo"].astype(bool)

df_tr = df_mod[df_mod["conjunto"] == "entrenamiento"]
df_ts = df_mod[(df_mod["conjunto"] == "prueba") & (~df_mod["estacion_nueva"])]

# 2. Baseline ingenuo: Persistencia (el estado actual predice el siguiente)
y_real = df_ts["target_en_riesgo"]
y_pred_naive = df_ts["en_riesgo"]
print("=== BASELINE: PERSISTENCIA (TEST) ===")
print(classification_report(y_real, y_pred_naive, digits=4))

# 3. Modelo Candidato: Regresión Logística balanceada
modelo_p2 = LogisticRegression(class_weight="balanced", random_state=RANDOM_SEED, max_iter=1000)
modelo_p2.fit(df_tr[P2_VARIABLES], df_tr["target_en_riesgo"])
y_pred_log = modelo_p2.predict(df_ts[P2_VARIABLES])
y_prob_log = modelo_p2.predict_proba(df_ts[P2_VARIABLES])[:, 1]
print("=== MODELO CANDIDATO: REGRESIÓN LOGÍSTICA (TEST) ===")
print(classification_report(y_real, y_pred_log, digits=4))

=== BASELINE: PERSISTENCIA (TEST) ===
              precision    recall  f1-score   support

       False     0.9807    0.9807    0.9807    719900
        True     0.8331    0.8334    0.8332     83293

    accuracy                         0.9654    803193
   macro avg     0.9069    0.9070    0.9070    803193
weighted avg     0.9654    0.9654    0.9654    803193

=== MODELO CANDIDATO: REGRESIÓN LOGÍSTICA (TEST) ===
              precision    recall  f1-score   support

       False     0.9823    0.9663    0.9743    719900
        True     0.7450    0.8496    0.7938     83293

    accuracy                         0.9542    803193
   macro avg     0.8636    0.9080    0.8841    803193
weighted avg     0.9577    0.9542    0.9556    803193



### 7. Registro de Experimentos y Trazabilidad en MLflow

**Gobierno del experimento:**
Para garantizar reproducibilidad y auditoría completa del ciclo de vida del modelo, se registran los parámetros, métricas y artefactos visuales en el servidor central de MLflow Tracking bajo el experimento institucional del equipo.

**Elementos registrados en el Run:**
- **Parámetros (`params`):** Configuración de la granularidad temporal (horaria), ventana operativa (corte a las 03:00 hrs), umbral de capacidad fijado al 50% (`P2_UMBRAL`), hiperparámetros del estimador y rangos de fechas de las particiones.
- **Métricas (`metrics`):** Desempeño sobre el conjunto de prueba (`test_f1`, `test_pr_auc`), asegurando que las estaciones nuevas incorporadas en mayo-junio de 2026 queden excluidas de la evaluación para evitar sesgos por arranque en frío.
- **Etiquetas de trazabilidad (`tags`):** Identificador del equipo, pregunta P2, commit hash exacto del repositorio Git (con detección de estado *dirty*) y hash SHA-256 de los datos procesados.
- **Artefactos (`artifacts`):** Generación y guardado de las figuras diagnósticas oficiales (`plots.figuras_p2`): la **Matriz de Confusión** (conteo de aciertos y errores de clasificación) y la **Curva Precision-Recall** (evaluación integral del trade-off de clasificación en datos desbalanceados).

In [51]:
import mlflow
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, precision_recall_curve, auc
from proyecto_mibici.config import URI_MLFLOW_TRACKING, MLFLOW_EXPERIMENT_NAME, P2_UMBRAL
from proyecto_mibici.modeling.tracking import armar_params_tags, registrar_run
from proyecto_mibici.plots import figuras_p2

mlflow.set_tracking_uri(URI_MLFLOW_TRACKING)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

splits_p2 = {"Train": df_tr, "Validación": df_ts, "Test": df_ts}
params_base, tags_base = armar_params_tags(
    "P2", splits_p2, extra_params={"umbral": P2_UMBRAL}, col_tiempo="hora"
)

# Generar matriz de confusión y curva Precision-Recall usando plots.py
figs = figuras_p2(y_real, y_pred_log, y_prob_log, "logistic_regression")

# Métricas formateadas para MLflow
precision, recall, _ = precision_recall_curve(y_real, y_prob_log)
pr_auc_val = auc(recall, precision)

tabla_met = pd.DataFrame([{
    "conjunto": "Test",
    "f1": float(classification_report(y_real, y_pred_log, output_dict=True)["True"]["f1-score"]),
    "pr_auc": float(pr_auc_val),
}])
tabla_vacia = pd.DataFrame(columns=["origen_id", "nombre_estacion", "MAE"])

# Registrar run en MLflow
run_id = registrar_run(
    run_name="p2-baseline-logistic-regression",
    params={**params_base, "modelo": "LogisticRegression"},
    tags={**tags_base, "version": "v1_logistic"},
    tabla=tabla_met,
    tabla_est=tabla_vacia,
    figuras=figs,
)

for fig in figs.values():
    display(fig)
    plt.close(fig)

print(f"Run de P2 registrado exitosamente en MLflow con ID: {run_id}")

🏃 View run p2-baseline-logistic-regression at: http://localhost:8080/#/experiments/6/runs/91cea54c2747495a9c9f15d13a69c362
🧪 View experiment at: http://localhost:8080/#/experiments/6


<Figure size 600x600 with 2 Axes>

<Figure size 700x600 with 1 Axes>

Run de P2 registrado exitosamente en MLflow con ID: 91cea54c2747495a9c9f15d13a69c362
